# Tugas - Deploy Project Sendiri

**Nama:** Radit Firansah  
**NIM:** 244107020196

Menerapkan pipeline deployment yang sama untuk model project sendiri.

## Tujuan

Menerapkan pipeline deployment (Flask + Docker + Hugging Face Spaces) yang sama seperti Praktikum 2, tetapi menggunakan **model project sendiri**.

## Langkah-langkah Deployment

1. **Siapkan hasil training**
   - File model `.h5` (model dengan akurasi terbaik).
   - File scaler `.pkl` (jika preprocessing memakai `StandardScaler`).

2. **Buat `app.py`**
   - Buat aplikasi Flask yang memuat model `.h5` dan `scaler.pkl`.
   - Buat fungsi `preprocess_image` yang **sama persis** dengan preprocessing saat training.
   - Tambahkan route `/` (form upload) dan `/predict` (hasil prediksi).
   - Jalankan di `host='0.0.0.0'`, `port=7860` (wajib untuk Hugging Face Spaces).

3. **Buat `requirements.txt`**
   - Daftar library yang dipakai, contoh: `flask`, `tensorflow-cpu`, `numpy`, `scikit-learn`, `scikit-image`, `opencv-python-headless`, `gunicorn`.

4. **Buat `Dockerfile`**
   - Gunakan base `python:3.9-slim`, install `libgl1` & `libglib2.0-0`, copy file, expose port `7860`, jalankan `python app.py`.

5. **Buat Hugging Face Space**
   - Login ke Hugging Face → New Space.
   - Isi Space Name, License, SDK: **Docker**, Template kosong, Visibility: **Public**.
   - Buat Access Token dengan permission **Write**.

6. **Upload dengan Git**
   ```
   git clone https://huggingface.co/spaces/USERNAME_ANDA/NAMA_SPACE
   # copy app.py, requirements.txt, Dockerfile, model .h5, scaler .pkl ke folder hasil clone
   cd NAMA_SPACE
   git add .
   git commit -m "Deploy aplikasi ke HF"
   git push
   ```
   - Username: username Hugging Face Anda.
   - Password: **Access Token** (bukan password akun).

7. **Uji hasil**
   - Buka halaman Space → tab **App**, tunggu status *Building* → *Running*.
   - Upload gambar dan pastikan prediksi berjalan.

## Demo Inferensi Lokal

Memuat model hasil Praktikum 1 dan memprediksi satu gambar test. Ini membuktikan pipeline inferensi yang akan dipakai di `app.py` sudah benar.

In [1]:
import glob
import pickle
import numpy as np
import cv2
import tensorflow as tf
from skimage.feature import hog

model = tf.keras.models.load_model('day_night_model.h5')
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

path = sorted(glob.glob('../data/images/test/night/*.jpg'))[0]
img = cv2.imread(path)
img = cv2.resize(img, (256, 256))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

fitur = hog(gray, orientations=9, pixels_per_cell=(8,8),
            cells_per_block=(2,2), block_norm='L2-Hys',
            visualize=False, feature_vector=True)
fitur = scaler.transform(fitur.reshape(1, -1))

prob = model.predict(fitur, verbose=0)[0][0]
hasil = "Day (Siang)" if prob > 0.5 else "Night (Malam)"

print("Gambar:", path)
print("Probabilitas:", prob)
print("Prediksi:", hasil)

I0000 00:00:1790788897.275455   35012 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790788897.275809   35012 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790788897.308330   35012 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790788898.217013   35012 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790788898.217296   35012 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


E0000 00:00:1790788899.217856   35012 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1790788899.218067   35069 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1790788899.234049   35012 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Gambar: ../data/images/test/night/20151101_072507.jpg
Probabilitas: 7.3473534e-09
Prediksi: Night (Malam)


## Kesimpulan

Pipeline deployment terdiri dari dua bagian utama: **artefak model** (`.h5` dan `.pkl`) yang dihasilkan dari tahap training, dan **kode serving** (`app.py`, `requirements.txt`, `Dockerfile`) yang membungkus model menjadi layanan web.

`app.py` wajib memakai preprocessing yang **identik** dengan saat training (resize 256x256, grayscale, HOG dengan parameter yang sama, lalu `scaler.transform`), karena perbedaan preprocessing akan membuat prediksi tidak valid.

Docker memastikan lingkungan server sama dengan lingkungan pengembangan, sedangkan Hugging Face Spaces menyediakan hosting gratis dengan port `7860`. Dengan alur `git add → commit → push`, model project sendiri dapat diakses publik melalui web tanpa perlu mengelola server secara manual.